## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab dengan GPU T4); `CLAUDE.md` Bagian 3.1, 3.3, dan 3.4.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v3")
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (catatan sesi pertama dipertahankan)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers"]}}
log = AKAR / "log" / "versi_package.json"
pertama = json.loads(log.read_text())["pertama"] if log.exists() else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
xlsx, koma = AKAR / "data" / "data_wonder_bybni_3.xlsx", lambda x: f"{x:.1f}".replace(".", ",")
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{xlsx.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<14}Versi")
print("\n".join(f"{k:<14}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, aktual, target in [
        ("berkas data ada", "ya" if xlsx.exists() else "tidak", "ya"),
        ("GPU", gpu, "Tesla T4"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif",
         "ya" if not _cuda_dulu or _cublas_dulu == ":4096:8" else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if aktual == target else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")